# 10-03 Agent 监控与追踪

**为什么需要监控**: Agent 的行为不可预测，必须追踪每步的输入/输出/延迟/成本。

**本节目标**：链路追踪、成本监控、LangSmith 集成概念

---

In [ ]:
import time, json
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TraceSpan:
    """追踪单元：记录一个节点的执行信息"""
    name: str
    start_time: float = field(default_factory=time.time)
    end_time: float = 0
    input_data: str = ""
    output_data: str = ""
    tokens_used: int = 0
    error: Optional[str] = None
    
    @property
    def latency_ms(self) -> float:
        return (self.end_time - self.start_time) * 1000

class AgentTracer:
    """Agent 全链路追踪器"""
    def __init__(self):
        self.spans: list[TraceSpan] = []
        self.total_cost = 0.0
    
    def trace(self, name: str, input_data: str = ""):
        span = TraceSpan(name=name, input_data=input_data[:200])
        self.spans.append(span)
        return span
    
    def end_span(self, span: TraceSpan, output: str = "", tokens: int = 0):
        span.end_time = time.time()
        span.output_data = output[:200]
        span.tokens_used = tokens
        self.total_cost += tokens * 0.15e-6  # GPT-4o-mini pricing estimate
    
    def summary(self):
        print(f"{'节点':<20} {'延迟(ms)':<10} {'Tokens':<8} {'状态'}")
        print("-" * 50)
        for s in self.spans:
            status = '❌' if s.error else '✅'
            print(f"{s.name:<20} {s.latency_ms:<10.0f} {s.tokens_used:<8} {status}")
        print(f"\n总计: {len(self.spans)}步, {sum(s.tokens_used for s in self.spans)} tokens, ${self.total_cost:.5f}")

# 演示
tracer = AgentTracer()

# 模拟 Agent 执行链路
for step_name, tokens in [("intent_classify", 50), ("rag_retrieve", 0), ("llm_generate", 300), ("compliance_check", 80)]:
    span = tracer.trace(step_name, input_data="用户问题...")
    time.sleep(0.05)  # 模拟处理
    tracer.end_span(span, output="处理结果...", tokens=tokens)

tracer.summary()

In [ ]:
# LangSmith 集成概念
print("""
LangSmith 监控平台（LangChain 官方）:

  1. 设置环境变量:
     LANGCHAIN_TRACING_V2=true
     LANGCHAIN_API_KEY=ls-xxx
     LANGCHAIN_PROJECT=bilibili-ad-agent

  2. 自动追踪: LangChain/LangGraph 的每次调用自动上报
     - 每个 chain.invoke() 的输入/输出/延迟
     - LLM 调用的 token 用量和成本
     - 工具调用的参数和结果
     - Agent 的完整决策链路

  3. 评估 (Evaluation):
     - 创建 Dataset (问题+标准答案)
     - 用 LLM-as-Judge 评估 Agent 输出
     - 跟踪指标趋势 (准确率/延迟/成本)

  4. 替代方案:
     - Phoenix (Arize): 开源，支持 traces + evals
     - Langfuse: 开源，自部署
     - 自建: OpenTelemetry + Grafana
""")

## 面试速记

| 问题 | 要点 |
|------|------|
| 为什么要监控 Agent | 非确定性行为、成本控制、质量漂移检测、故障排查 |
| 监控哪些指标 | 延迟、token消耗、成本、工具调用成功率、用户满意度 |
| LangSmith 的核心功能 | Tracing(链路追踪)、Evaluation(评估)、Dataset(测试集管理) |